# G2: Predicted-Analysis Generation

This notebook is the G2 generation entry point. It accepts a reflection document and segment analysis produced by `02_segment_inference.ipynb`, then calls the shared assessment-and-feedback runner. Run inference for the same document first and save its predicted-analysis artifact under approved protected storage. Never substitute G3 human labels.

Run from the `prebi_v1` project root with the `.venv` kernel. The current sample document source is the provisional test split; use it only for an approved pilot or final test, not for prompt or model tuning. Never print, commit, or send protected reflection data or predictions to an unapproved provider or tracing workspace.

## 1. Detect Current Notebook File Path

Jupyter does not reliably expose the active notebook path to Python. Run from the project root and set the notebook paths explicitly.

In [14]:
from pathlib import Path

WORKSPACE_DIR = Path.cwd().resolve()
assert WORKSPACE_DIR.name == "prebi_v1", "Run this notebook from the prebi_v1 project root."

## G2 Generation Workflow

### Load R1 reflection segments without gold labels

This cell loads all configured R1 development documents using only `document_id`, `segment_id`, `text`, and `segment_order`. Candidate copies are deduplicated only after verifying identical text and order; no annotation targets are loaded.

In [15]:
import os

import pandas as pd
from dotenv import load_dotenv
from reflection_assessment_feedback import EXPERT_RUBRIC, ReflectionDocument, ReflectionSegment
from reflection_assessment_feedback.experiment_config import load_experiment_config

WORKSPACE_DIR = Path.cwd().resolve()
assert WORKSPACE_DIR.name == "prebi_v1", "Run this notebook from the prebi_v1 project root."
load_dotenv(WORKSPACE_DIR / ".env", override=False)
EXPERIMENT_CONFIG = load_experiment_config()
DATASET_CONFIG = EXPERIMENT_CONFIG["dataset"]
SPLIT_PATH = (
    WORKSPACE_DIR
    / EXPERIMENT_CONFIG["paths"]["provisional_split_directory"]
    / f"provisional_{DATASET_CONFIG['prediction_split']}_modeling_wide.csv"
)
FEEDBACK_PATH = (WORKSPACE_DIR / EXPERIMENT_CONFIG["paths"]["human_feedback"]).resolve()
feedback_rows = pd.read_csv(
    FEEDBACK_PATH,
    usecols=["essay_name", "document_id"],
    dtype={"document_id": str},
)
feedback_rows["essay_name"] = feedback_rows["essay_name"].astype(str).str.strip().str.upper()
feedback_rows["document_id"] = feedback_rows["document_id"].astype(str).str.strip()
split_document_ids = set(
    pd.read_csv(SPLIT_PATH, usecols=["document_id"], dtype={"document_id": str})[
        "document_id"
    ].dropna()
)
DEVELOPMENT_DOCUMENT_IDS = sorted(
    set(
        feedback_rows.loc[
            feedback_rows["essay_name"] == DATASET_CONFIG["development_essay_name"],
            "document_id",
        ]
    )
    & split_document_ids
)
if not DEVELOPMENT_DOCUMENT_IDS:
    raise ValueError("No development-cohort documents are available in the selected split.")

reflection_rows = pd.read_csv(
    SPLIT_PATH,
    usecols=["document_id", "segment_id", "text", "segment_order"],
    dtype={"document_id": str, "segment_id": str},
)
reflection_rows["text"] = reflection_rows["text"].fillna("").astype(str)
reflection_rows["segment_order"] = pd.to_numeric(
    reflection_rows["segment_order"], errors="raise"
).astype(int)
reflection_rows = reflection_rows.loc[
    reflection_rows["document_id"].isin(DEVELOPMENT_DOCUMENT_IDS)
].copy()
if reflection_rows[["segment_id", "text", "segment_order"]].isna().any().any():
    raise ValueError("Selected development documents contain missing IDs, text, or order.")

DEVELOPMENT_DOCUMENTS = {}
for document_id, candidate_rows in reflection_rows.groupby("document_id", sort=True):
    consistency = candidate_rows.groupby("segment_id").agg(
        text_variants=("text", "nunique"),
        order_variants=("segment_order", "nunique"),
    )
    if (consistency > 1).any().any():
        raise ValueError("Candidate copies disagree on segment text or source order.")
    document_rows = (
        candidate_rows.drop_duplicates("segment_id")
        .sort_values("segment_order")
        .reset_index(drop=True)
    )
    DEVELOPMENT_DOCUMENTS[document_id] = ReflectionDocument(
        document_id=document_id,
        segments=[
            ReflectionSegment(
                segment_id=row.segment_id,
                text=row.text,
                order=row.segment_order,
            )
            for row in document_rows.itertuples(index=False)
        ],
    )
if set(DEVELOPMENT_DOCUMENTS) != set(DEVELOPMENT_DOCUMENT_IDS):
    raise ValueError("Some configured development documents have no segment rows.")
print(
    f"Selected {len(DEVELOPMENT_DOCUMENTS)} R1 development documents and "
    f"{sum(len(document.segments) for document in DEVELOPMENT_DOCUMENTS.values())} ordered segments; text withheld."
)

Selected 6 R1 development documents and 282 ordered segments; text withheld.


### Load predicted analyses: GBERT or TF-IDF

GBERT remains the default: run `02_segment_inference.ipynb` first. For the CPU-only ML backend, train and save a trusted model with `00_segment_classification_ml.ipynb`, set `PREDICTION_BACKEND='tfidf'` and `ML_MODEL_PATH` below, then enable `WRITE_ML_PREDICTIONS` to create local predictions. No provider calls occur in this step.

TF-IDF predictions are isolated under protected `g2/predictions/tfidf/<model-sha256>/`; GBERT predictions are unchanged. Validation checks config, model and document fingerprints, prediction source, and segment coverage. The downstream generator records the prediction-file hash, keeping runs distinguishable. Do not pool results across classifier backends or model versions.

In [ ]:
import hashlib
import json
from urllib.parse import quote

from reflection_assessment_feedback import IntermediateAnalysis, make_generation_request
from reflection_assessment_feedback.experiment_config import experiment_config_sha256
from reflection_assessment_feedback.segment_classification_ml import (
    SegmentClassifierML, export_g2_prediction,
)

PREDICTION_BACKEND = 'gbert'
ML_MODEL_PATH = None
WRITE_ML_PREDICTIONS = False

protected_root_value = os.getenv('PREBI_DATA_ROOT')
if not protected_root_value:
    raise RuntimeError('Set PREBI_DATA_ROOT to approved protected storage before loading G2 predictions.')
PROTECTED_ROOT = Path(protected_root_value).expanduser().resolve()
if PROTECTED_ROOT == WORKSPACE_DIR or WORKSPACE_DIR in PROTECTED_ROOT.parents:
    raise ValueError('PREBI_DATA_ROOT must be outside the project source tree.')
PREDICTION_DIR = PROTECTED_ROOT / 'g2' / 'predictions'
if PREDICTION_BACKEND not in {'gbert', 'tfidf'}:
    raise ValueError('Choose gbert or tfidf for PREDICTION_BACKEND.')
if PREDICTION_BACKEND == 'tfidf':
    if ML_MODEL_PATH is None:
        raise ValueError('Set ML_MODEL_PATH to a trusted model saved by the ML notebook.')
    ml_model_path = Path(ML_MODEL_PATH).expanduser().resolve()
    if PROTECTED_ROOT not in ml_model_path.parents:
        raise ValueError('The ML model must be stored under PREBI_DATA_ROOT.')
    ml_model_sha256 = hashlib.sha256(ml_model_path.read_bytes()).hexdigest()
    PREDICTION_DIR = PREDICTION_DIR / 'tfidf' / ml_model_sha256
    if WRITE_ML_PREDICTIONS:
        ml_classifier = SegmentClassifierML.load(ml_model_path)
        for document in DEVELOPMENT_DOCUMENTS.values():
            export_g2_prediction(
                ml_classifier, document, protected_root=PROTECTED_ROOT,
                project_root=WORKSPACE_DIR, config_sha256=experiment_config_sha256(),
                model_sha256=ml_model_sha256,
            )
PREDICTED_ANALYSES = {}
PREDICTION_PATHS = {}
for document_id, document in DEVELOPMENT_DOCUMENTS.items():
    prediction_path = PREDICTION_DIR / f"predicted-analysis-{quote(document_id, safe='')}.json"
    if not prediction_path.is_file():
        raise FileNotFoundError('A development document is missing its prediction artifact.')
    prediction_package = json.loads(prediction_path.read_text(encoding='utf-8'))
    if prediction_package.get('document_id') != document_id:
        raise ValueError('Prediction artifact does not match its selected document.')
    if prediction_package.get('experiment_config_sha256') != experiment_config_sha256():
        raise ValueError('Prediction artifact config fingerprint does not match this experiment.')
    if PREDICTION_BACKEND == 'tfidf':
        if prediction_package.get('classifier_backend') != 'tfidf':
            raise ValueError('Prediction artifact has the wrong classifier backend.')
        if prediction_package.get('classifier_model_sha256') != ml_model_sha256:
            raise ValueError('Prediction artifact has the wrong ML model fingerprint.')
        document_sha256 = hashlib.sha256(document.model_dump_json().encode()).hexdigest()
        if prediction_package.get('document_sha256') != document_sha256:
            raise ValueError('ML prediction artifact refers to different document text or order.')
    predicted_analysis = IntermediateAnalysis.model_validate(prediction_package['analysis'])
    if predicted_analysis.source != 'predicted':
        raise ValueError('G2 requires predicted, not human, segment analysis.')
    make_generation_request(
        condition='G2', document=document, rubric=EXPERT_RUBRIC, analysis=predicted_analysis,
    )
    PREDICTED_ANALYSES[document_id] = predicted_analysis
    PREDICTION_PATHS[document_id] = prediction_path
print(f'Loaded and validated {PREDICTION_BACKEND} predictions for {len(PREDICTED_ANALYSES)} development documents.')

Loaded and validated predictions for 6 development documents.


### Generate and persist the R1 G2 batch

The shared `config/experiment.toml` supplies the model, repetition count, and LangSmith tracing preference. Provider approval remains an explicit runtime gate. Each G2 run is matched to that document's G1/G3 artifact and predicted-analysis hash, paced through the shared Gemini limiter, and persisted under protected storage. Traces may contain the full reflection, prompt, and output.

In [17]:
import hashlib
import importlib
import json
import os
import tempfile
from datetime import datetime, timezone
from pathlib import Path

from dotenv import dotenv_values, load_dotenv
from langsmith import tracing_context
from reflection_assessment_feedback.experiment_config import (
    experiment_config_sha256,
    load_experiment_config,
)
from reflection_assessment_feedback.rate_limit import reserve_request_slot

EXPERIMENT_CONFIG = load_experiment_config()
GENERATION_CONFIG = EXPERIMENT_CONFIG["generation"]
langsmith_dotenv_path = WORKSPACE_DIR / ".env"
langsmith_dotenv = dotenv_values(langsmith_dotenv_path)
load_dotenv(langsmith_dotenv_path, override=False)
conflicting_langsmith_settings = [
    name
    for name in ("LANGSMITH_API_KEY", "LANGSMITH_PROJECT", "LANGSMITH_ENDPOINT")
    if langsmith_dotenv.get(name)
    and os.getenv(name)
    and langsmith_dotenv[name] != os.getenv(name)
]
if conflicting_langsmith_settings:
    raise RuntimeError(
        "LangSmith settings differ between .env and the active kernel environment: "
        f"{', '.join(conflicting_langsmith_settings)}. The kernel environment takes "
        "precedence; update or unset the conflicting value and restart the kernel. "
        "Secret values are not displayed."
    )

from langchain_google_genai import ChatGoogleGenerativeAI
import reflection_assessment_feedback.generator as generation_module
from reflection_assessment_feedback import make_generation_request

generation_module = importlib.reload(generation_module)
RubricGenerationRunner = generation_module.RubricGenerationRunner
PROMPT_VERSION = generation_module.PROMPT_VERSION
if PROMPT_VERSION != GENERATION_CONFIG["expected_prompt_version"]:
    raise ValueError("Code prompt version differs from experiment config.")
if EXPERT_RUBRIC.version != GENERATION_CONFIG["expected_rubric_version"]:
    raise ValueError("Rubric version differs from experiment config.")

APPROVE_EXTERNAL_PROCESSING = True
ENABLE_LANGSMITH_TRACING = GENERATION_CONFIG["enable_langsmith_tracing"]
REPETITIONS = GENERATION_CONFIG["repetitions"]
assert APPROVE_EXTERNAL_PROCESSING, "Confirm provider processing is approved before running."
assert ENABLE_LANGSMITH_TRACING, "Tracing is required for this approved development batch."
assert len(DEVELOPMENT_DOCUMENTS) == len(PREDICTED_ANALYSES) == 6
assert os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY"), (
    "Configure GOOGLE_API_KEY or GEMINI_API_KEY in the approved environment."
)
assert os.getenv("LANGSMITH_API_KEY"), (
    "Configure LANGSMITH_API_KEY in the local .env or approved environment."
)
assert os.getenv("LANGSMITH_PROJECT"), (
    "Configure LANGSMITH_PROJECT in the local .env or approved environment."
)
assert os.getenv("LANGSMITH_ENDPOINT", "https://eu.api.smith.langchain.com").rstrip("/") == (
    "https://eu.api.smith.langchain.com"
), "Tracing is restricted to the LangSmith EU endpoint."

PROTECTED_ROOT = Path(os.environ["PREBI_DATA_ROOT"]).expanduser().resolve()
if PROTECTED_ROOT == WORKSPACE_DIR or WORKSPACE_DIR in PROTECTED_ROOT.parents:
    raise ValueError("PREBI_DATA_ROOT must be outside the project source tree.")
COMPARISON_DIRECTORY = PROTECTED_ROOT / "g1-g3" / "demo-runs"
G2_RUN_DIRECTORY = PROTECTED_ROOT / "g2" / "demo-runs"
G2_RUN_DIRECTORY.mkdir(parents=True, exist_ok=True, mode=0o700)
if not G2_RUN_DIRECTORY.resolve().is_relative_to(PROTECTED_ROOT):
    raise ValueError("G2 run storage must resolve within PREBI_DATA_ROOT.")
RATE_LIMIT_STATE = PROTECTED_ROOT / ".rate_limits" / "gemini.state"
CONFIG_SHA256 = experiment_config_sha256()

existing_runs = []
for run_path in G2_RUN_DIRECTORY.glob("g2-run-*.json"):
    envelope = json.loads(run_path.read_text(encoding="utf-8"))
    run = envelope.get("run", {})
    existing_runs.append((envelope, run))

PREPARED_DOCUMENTS = []
for document_id, document in DEVELOPMENT_DOCUMENTS.items():
    if document_id not in PREDICTED_ANALYSES:
        raise ValueError("A development document is missing predicted analysis.")
    predicted_analysis = PREDICTED_ANALYSES[document_id]
    if predicted_analysis.source != "predicted":
        raise ValueError("G2 requires predicted segment analysis.")
    if len(predicted_analysis.segments) != len(document.segments):
        raise ValueError("G2 prediction coverage differs from its source document.")
    request = make_generation_request(
        condition="G2",
        document=document,
        rubric=EXPERT_RUBRIC,
        analysis=predicted_analysis,
    )

    matching_pairs = []
    for comparison_path in COMPARISON_DIRECTORY.glob("comparison-*.json"):
        comparison_package = json.loads(comparison_path.read_text(encoding="utf-8"))
        if (
            comparison_package.get("document_id") != document_id
            or comparison_package.get("rubric_version") != EXPERT_RUBRIC.version
            or comparison_package.get("experiment_config_sha256") != CONFIG_SHA256
        ):
            continue
        for pair in comparison_package.get("pairs", []):
            g1_run = pair.get("G1", {})
            g3_run = pair.get("G3", {})
            if (
                g1_run.get("condition") == "G1"
                and g3_run.get("condition") == "G3"
                and g1_run.get("repetition") == g3_run.get("repetition") == pair.get("repetition")
                and g1_run.get("repetition") <= REPETITIONS
                and g1_run.get("model_name") == g3_run.get("model_name") == GENERATION_CONFIG["model_name"]
                and g1_run.get("prompt_version") == g3_run.get("prompt_version") == PROMPT_VERSION
            ):
                matching_pairs.append((comparison_path, comparison_package, pair))
    for repetition in range(1, REPETITIONS + 1):
        candidates = [
            item for item in matching_pairs
            if item[2].get("repetition") == repetition
        ]
        if not candidates:
            raise FileNotFoundError(
                "A development document has no config-matched G1/G3 pair; no G2 calls were made."
            )
        comparison_path, comparison_package, pair = max(
            candidates,
            key=lambda item: item[0].stat().st_mtime,
        )
        comparison_sha256 = hashlib.sha256(comparison_path.read_bytes()).hexdigest()
        already_saved = any(
            run.get("document_id") == document_id
            and run.get("condition") == "G2"
            and run.get("repetition") == repetition
            and run.get("model_name") == GENERATION_CONFIG["model_name"]
            and run.get("prompt_version") == PROMPT_VERSION
            and run.get("rubric_version") == EXPERT_RUBRIC.version
            and run.get("analysis_reference_id", "").endswith(
                hashlib.sha256(PREDICTION_PATHS[document_id].read_bytes()).hexdigest()
            )
            and envelope.get("paired_g1_g3_comparison_sha256") == comparison_sha256
            for envelope, run in existing_runs
        )
        PREPARED_DOCUMENTS.append(
            {
                "document_id": document_id,
                "document": document,
                "request": request,
                "repetition": repetition,
                "comparison_sha256": comparison_sha256,
                "prediction_sha256": hashlib.sha256(
                    PREDICTION_PATHS[document_id].read_bytes()
                ).hexdigest(),
                "already_saved": already_saved,
            }
        )

chat_model = ChatGoogleGenerativeAI(
    model=GENERATION_CONFIG["model_name"],
    temperature=GENERATION_CONFIG["temperature"],
)
runner = RubricGenerationRunner(
    chat_model,
    enable_langsmith_tracing=ENABLE_LANGSMITH_TRACING,
)
G2_RUN_RECORDS = []
for prepared in PREPARED_DOCUMENTS:
    if prepared["already_saved"]:
        continue
    reserve_request_slot(
        RATE_LIMIT_STATE,
        GENERATION_CONFIG["minimum_request_interval_seconds"],
    )
    with tracing_context(enabled=ENABLE_LANGSMITH_TRACING):
        generated_run = runner.generate(
            prepared["request"],
            repetition=prepared["repetition"],
        )

    dimensions = {
        assessment.dimension_id: {
            "score": float(assessment.band),
            "justification": assessment.justification,
            "evidence_segment_ids": assessment.evidence_segment_ids,
        }
        for assessment in generated_run.output.assessment.component_assessments
    }
    if set(dimensions) != {"SW", "UA", "HA"}:
        raise ValueError("G2 assessment must contain exactly the three rubric dimensions.")
    run_record = {
        "run_id": generated_run.run_id,
        "document_id": prepared["document_id"],
        "condition": "G2",
        "repetition": generated_run.repetition,
        "model_provider": GENERATION_CONFIG["provider"],
        "model_name": generated_run.model_name,
        "model_version": None,
        "prompt_version": generated_run.prompt_version,
        "rubric_version": EXPERT_RUBRIC.version,
        "generation_parameters": generated_run.generation_parameters,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "analysis_source": "predicted",
        "analysis_reference_id": f"predicted-analysis-sha256-{prepared['prediction_sha256']}",
        "assessment": {"dimensions": dimensions},
        "feedback": {
            component: [
                {"text": point.message, "evidence_segment_ids": point.evidence_segment_ids}
                for point in getattr(generated_run.output.feedback, component)
            ]
            for component in ("strengths", "weaknesses", "suggestions")
        },
        "validation_status": "valid",
        "validation_message": None,
    }
    envelope = {
        "schema_version": "1.0.0",
        "record_type": "g2_generation_run",
        "experiment_config_sha256": CONFIG_SHA256,
        "run": run_record,
        "paired_g1_g3_comparison_sha256": prepared["comparison_sha256"],
    }
    run_path = G2_RUN_DIRECTORY / f"g2-run-{generated_run.run_id}.json"
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", dir=G2_RUN_DIRECTORY,
            prefix=".g2-run-", suffix=".tmp", delete=False,
        ) as temporary_file:
            temporary_path = Path(temporary_file.name)
            json.dump(envelope, temporary_file, ensure_ascii=False, indent=2, default=str)
            temporary_file.write("\n")
        os.chmod(temporary_path, 0o600)
        os.replace(temporary_path, run_path)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)
    G2_RUN_RECORDS.append(run_record)

expected_records = len(PREPARED_DOCUMENTS)
existing_matching_count = sum(prepared["already_saved"] for prepared in PREPARED_DOCUMENTS)
if len(G2_RUN_RECORDS) + existing_matching_count != expected_records:
    raise ValueError("Not every prepared G2 development run is persisted.")
print(
    f"R1 G2 batch complete: {expected_records} document/repetition records; "
    f"newly generated={len(G2_RUN_RECORDS)}, reused={existing_matching_count}; "
    f"tracing={'on' if ENABLE_LANGSMITH_TRACING else 'off'}."
)


/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_gen

R1 G2 batch complete: 6 document/repetition records; newly generated=6, reused=0; tracing=on.


### Finalize prompt provenance in persisted G2 records

Record the registered prompt-file hash and exact rendered-prompt hash from each completed run. This updates only metadata for runs generated immediately above; it makes no provider calls.

In [ ]:
import json
import os
import tempfile
from pathlib import Path

from reflection_assessment_feedback.prompt_registry import resolve_prompt
from reflection_assessment_feedback.rubric import RUBRIC_ARTIFACT

registered_prompt = resolve_prompt(
    GENERATION_CONFIG["prompt_id"],
    PROMPT_VERSION,
)
for generated_run in g2_runs:
    run_record = next(
        record for record in G2_RUN_RECORDS
        if record["run_id"] == generated_run.run_id
    )
    if generated_run.prompt_artifact_sha256 != registered_prompt.sha256:
        raise ValueError("Generated run prompt hash differs from the registered prompt artifact.")
    if generated_run.rubric_artifact_sha256 != RUBRIC_ARTIFACT.sha256:
        raise ValueError("Generated run rubric hash differs from the configured rubric artifact.")
    run_record["prompt_artifact_sha256"] = generated_run.prompt_artifact_sha256
    run_record["prompt_components"] = generated_run.prompt_components
    run_record["rendered_prompt_sha256"] = generated_run.rendered_prompt_sha256
    run_record["rubric_artifact_sha256"] = generated_run.rubric_artifact_sha256
    run_path = G2_RUN_DIRECTORY / f"g2-run-{generated_run.run_id}.json"
    envelope = json.loads(run_path.read_text(encoding="utf-8"))
    envelope["run"]["prompt_artifact_sha256"] = generated_run.prompt_artifact_sha256
    envelope["run"]["prompt_components"] = generated_run.prompt_components
    envelope["run"]["rendered_prompt_sha256"] = generated_run.rendered_prompt_sha256
    envelope["run"]["rubric_artifact_sha256"] = generated_run.rubric_artifact_sha256
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", dir=G2_RUN_DIRECTORY,
            prefix=".g2-run-provenance-", suffix=".tmp", delete=False,
        ) as temporary_file:
            temporary_path = Path(temporary_file.name)
            json.dump(envelope, temporary_file, ensure_ascii=False, indent=2)
            temporary_file.write("\n")
        os.chmod(temporary_path, 0o600)
        os.replace(temporary_path, run_path)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)
print(f"Prompt and rubric provenance finalized for {len(g2_runs)} newly generated G2 runs.")